# Soft Meta Chatterbox TTS Server

This notebook installs only the `soft-meta` repositories and the official Resemble AI Chatterbox dependency. Select an **L4 GPU** before running all cells.


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq ffmpeg libsndfile1 git curl
cd /content
rm -rf bin
curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba
/content/bin/micromamba create -y -n sm311 -c conda-forge python=3.11 pip


In [ ]:
%%bash
set -euo pipefail
MM=/content/bin/micromamba
cd /content
rm -rf chatterbox-v2 Chatterbox-TTS-Server

git clone --branch v0.1.0 --depth 1 https://github.com/soft-meta/chatterbox-v2.git
git clone --branch v0.1.0 --depth 1 https://github.com/soft-meta/Chatterbox-TTS-Server.git

$MM run -n sm311 python -m pip install -U pip setuptools wheel
$MM run -n sm311 python -m pip install --index-url https://download.pytorch.org/whl/cu124 torch==2.6.0 torchaudio==2.6.0
$MM run -n sm311 python -m pip install -e /content/chatterbox-v2
$MM run -n sm311 python -m pip install -r /content/Chatterbox-TTS-Server/requirements-colab.txt


In [ ]:
import os, signal, socket, subprocess, time
from pathlib import Path
from IPython.display import HTML, display

PORT = 8004
PROJECT = Path('/content/Chatterbox-TTS-Server')
LOG = Path('/content/soft_meta_chatterbox.log')
MM = '/content/bin/micromamba'

subprocess.run(f"lsof -t -i:{PORT} | xargs -r kill -9", shell=True, check=False)
LOG.unlink(missing_ok=True)
log_handle = LOG.open('w', encoding='utf-8')
process = subprocess.Popen(
    [MM, 'run', '-n', 'sm311', 'python', '-u', 'start.py'],
    cwd=PROJECT,
    stdout=log_handle,
    stderr=subprocess.STDOUT,
    env={**os.environ, 'PYTHONUNBUFFERED': '1'},
    start_new_session=True,
)

def open_port():
    try:
        with socket.create_connection(('127.0.0.1', PORT), timeout=.5):
            return True
    except OSError:
        return False

for _ in range(240):
    if process.poll() is not None:
        log_handle.flush()
        raise RuntimeError(LOG.read_text(errors='replace')[-12000:])
    if open_port():
        break
    time.sleep(2)
else:
    raise TimeoutError('Server did not start. Check the log cell.')

from google.colab.output import eval_js
url = eval_js(f'google.colab.kernel.proxyPort({PORT})')
display(HTML(f'<a href="{url}" target="_blank" style="display:inline-block;padding:12px 18px;background:#6657e8;color:white;border-radius:8px;text-decoration:none;font-weight:700">Open Soft Meta Chatterbox TTS Server</a>'))
print('Server PID:', process.pid)
print('Log:', LOG)


In [ ]:
from pathlib import Path
log = Path('/content/soft_meta_chatterbox.log')
print(log.read_text(errors='replace')[-15000:] if log.exists() else 'No log yet.')
